<a href="https://colab.research.google.com/github/Guliko24/PubMed_Fetch/blob/main/Week1_Day4_Hybrid_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip -q install sentence-transformers rank-bm25 numpy

In [2]:
from pathlib import Path
import subprocess
import sys

GITHUB_USERNAME = "Guliko24"
REPO_NAME = "PubMed_Fetch"

repo_dir = Path("/content") / REPO_NAME

if not repo_dir.exists():
    subprocess.run(
        [
            "git",
            "clone",
            f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git",
            str(repo_dir),
        ],
        check=True,
    )
else:
    print("Repository already cloned.")
    print(
        "If you updated search_utils.py on GitHub since cloning, "
        "pull it before importing."
    )

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

!git -C /content/PubMed_Fetch pull
import search_utils

print("Utility imported from:", search_utils.__file__)
assert hasattr(
    search_utils, "reciprocal_rank_fusion"
), "RRF function missing: pull your latest GitHub changes and restart runtime."

Already up to date.
Utility imported from: /content/PubMed_Fetch/search_utils.py


In [3]:
DATA_FILE = repo_dir / "pubmed_abstracts.json"

# If your Day 1 JSON is in Drive rather than GitHub, mount Drive and
# replace DATA_FILE with its actual Drive path.

documents = search_utils.load_pubmed_data(DATA_FILE)

pmids = [doc["pmid"] for doc in documents]
assert len(pmids) == len(set(pmids)), "Duplicate PMIDs in the JSON file."

print("Documents loaded:", len(documents))
print("First PMID:", documents[0]["pmid"])

Documents loaded: 10
First PMID: 34582785


In [4]:
import hashlib
import json
import numpy as np

fingerprint_input = json.dumps(
    {
        "model": search_utils.MODEL_NAME,
        "documents": [
            {
                "pmid": doc["pmid"],
                "title": doc["title"],
                "abstract": doc["abstract"],
            }
            for doc in documents
        ],
    },
    sort_keys=True,
).encode("utf-8")

fingerprint = hashlib.sha256(fingerprint_input).hexdigest()[:16]
embedding_file = repo_dir / f"day3_embeddings_{fingerprint}.npy"

print("Expected embeddings file:", embedding_file.name)

if not embedding_file.exists():
    raise FileNotFoundError(
        f"Could not find {embedding_file.name}. Check that you uploaded "
        "your Day 3 file to GitHub and cloned/pulled the latest commit. "
        "If your Day 3 code used a different filename scheme, do not "
        "rename an arbitrary .npy file: regenerate embeddings from "
        "these documents using search_utils.generate_embeddings()."
    )

doc_embeddings = np.load(embedding_file, allow_pickle=False)

assert doc_embeddings.ndim == 2
assert doc_embeddings.shape[0] == len(documents)
assert np.isfinite(doc_embeddings).all()

print("Embedding shape:", doc_embeddings.shape)

Expected embeddings file: day3_embeddings_3071cbfb6ef3dc37.npy
Embedding shape: (10, 384)


In [5]:
# Run only if the preceding cell could not verify your saved embeddings:
#
# texts = [f"{d['title']}. {d['abstract']}" for d in documents]
# doc_embeddings = search_utils.generate_embeddings(texts)
# np.save(embedding_file, doc_embeddings)

RUN BOTH SEARCHES

In [6]:
QUERY = "human brain cell types identified by single-cell transcriptomics"

# Retrieve the whole small corpus into each ranking before fusion.
candidate_count = len(documents)

bm25_index = search_utils.build_bm25_index(documents)

bm25_results = search_utils.search_bm25(
    bm25_index,
    documents,
    QUERY,
    top_k=candidate_count,
)

vector_results = search_utils.search_vectors(
    QUERY,
    doc_embeddings,
    documents,
    top_k=candidate_count,
)

assert len(bm25_results) == candidate_count
assert len(vector_results) == candidate_count

print("BM25 candidates:", len(bm25_results))
print("Vector candidates:", len(vector_results))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

BM25 candidates: 10
Vector candidates: 10


With your small corpus, fusing both full ranked lists avoids excluding a paper before RRF gets a chance to score it. At a larger scale, you would choose a candidate window for each retriever and measure how that affects retrieval quality.

Fuse and Inspect

In [7]:
hybrid_results = search_utils.reciprocal_rank_fusion(
    bm25_results,
    vector_results,
    top_k=min(10, len(documents)),
    rank_constant=60,
)

print("QUERY:", QUERY)
print("-" * 100)

for position, result in enumerate(hybrid_results, start=1):
    print(
        f"{position:>2}. PMID {result['pmid']} | "
        f"RRF {result['rrf_score']:.5f} | "
        f"BM25 rank {result['bm25_rank']} | "
        f"Vector rank {result['vector_rank']}"
    )
    print("    ", result["doc"]["title"])

QUERY: human brain cell types identified by single-cell transcriptomics
----------------------------------------------------------------------------------------------------
 1. PMID 41053013 | RRF 0.03279 | BM25 rank 1 | Vector rank 1
     Mapping human brain cell type origin and diseases through single-cell transcriptomics.
 2. PMID 34582785 | RRF 0.03226 | BM25 rank 2 | Vector rank 2
     Single-nucleus transcriptome analysis reveals cell-type-specific molecular signatures across reward circuitry in the human brain.
 3. PMID 40585969 | RRF 0.03125 | BM25 rank 4 | Vector rank 4
     Single-cell transcriptomics of vascularized human brain organoids decipher lineage-specific stress adaptation in fetal hypoxia-reoxygenation injury.
 4. PMID 41456076 | RRF 0.03102 | BM25 rank 6 | Vector rank 3
     Exploring cellular heterogeneity: single-cell and spatial transcriptomics of Alzheimer's disease brains and iPSC-derived microglia.
 5. PMID 40701154 | RRF 0.03080 | BM25 rank 3 | Vector rank 7

Compare the TOP 5

In [8]:
print(f"{'Rank':<6} {'BM25 PMID':<16} {'Vector PMID':<16} {'Hybrid PMID':<16}")
print("-" * 58)

for i in range(min(5, len(documents))):
    print(
        f"{i + 1:<6} "
        f"{bm25_results[i]['doc']['pmid']:<16} "
        f"{vector_results[i]['doc']['pmid']:<16} "
        f"{hybrid_results[i]['pmid']:<16}"
    )

Rank   BM25 PMID        Vector PMID      Hybrid PMID     
----------------------------------------------------------
1      41053013         41053013         41053013        
2      34582785         34582785         34582785        
3      40701154         41456076         40585969        
4      40585969         40585969         41456076        
5      11847215         36544231         40701154        
